In [1]:
import torch
from tqdm import tqdm
from torch_geometric.data import Data, DataLoader
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch_geometric.nn import GCNConv, GATConv, GlobalAttention
from sklearn.metrics import f1_score, classification_report
import numpy as np

# 1. Modification du modèle pour mieux capturer les patterns minoritaires
class ImprovedGraphClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        # Couches convolutives avec normalisation
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.att_conv = GATConv(hidden_dim, hidden_dim, heads=2, concat=False)
        self.bn2 = nn.BatchNorm1d(hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        
        # Pooling hiérarchique avec attention renforcée
        self.pool = GlobalAttention(nn.Sequential(
            nn.Linear(hidden_dim, 2*hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(2*hidden_dim, 1)
        ))
        
        # Classificateur avec régularisation renforcée
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 2*hidden_dim),
            nn.BatchNorm1d(2*hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.6),
            nn.Linear(2*hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.4),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        x = F.relu(self.bn1(self.conv1(x, edge_index)))
        x = F.relu(self.bn2(self.att_conv(x, edge_index)))
        x = F.relu(self.conv2(x, edge_index))
        x = self.pool(x, batch)
        return self.classifier(x)

# 2. Loss Function adaptée avec focal loss et class balancing
class BalancedFocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=3, pos_weights=None):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.pos_weights = pos_weights  # [num_classes]

    def forward(self, inputs, targets):
        BCE_loss = F.binary_cross_entropy_with_logits(
            inputs, targets, 
            reduction='none', 
            pos_weight=self.pos_weights
        )
        
        pt = torch.exp(-BCE_loss)
        focal_loss = self.alpha * (1-pt)**self.gamma * BCE_loss
        
        # Masquer les classes sans positifs
        valid_mask = (targets.sum(dim=0) > 0)
        focal_loss = focal_loss[:, valid_mask].mean()
        
        return focal_loss

# 3. Calcul des poids de classe avec clipping
label_counts = df.drop(columns='name').sum(axis=0)
pos_weights = (len(df) - label_counts) / (label_counts + 1e-6)
pos_weights = np.clip(pos_weights, a_min=0, a_max=50)  # Clipping important!
pos_weights = torch.tensor(pos_weights.values, dtype=torch.float).to(device)

# 4. Initialisation avec poids Xavier améliorée
def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.kaiming_normal_(m.weight, nonlinearity='leaky_relu')
        if m.bias is not None:
            nn.init.constant_(m.bias, 0)

model = ImprovedGraphClassifier(input_dim=10, hidden_dim=256, output_dim=453)
model.apply(init_weights)
model = model.to(device)

# 5. Optimiseur avec gradient clipping
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
loss_fn = BalancedFocalLoss(alpha=0.8, gamma=3, pos_weights=pos_weights)

# 6. Fonction d'évaluation améliorée avec seuil adaptatif
def evaluate(loader):
    model.eval()
    preds, truths = [], []
    
    with torch.no_grad():
        for batch in loader:
            out = model(batch)
            preds.append(torch.sigmoid(out).cpu().numpy())
            truths.append(batch.y.cpu().numpy())

    preds = np.vstack(preds)
    truths = np.vstack(truths)
    
    # Calcul de seuils par classe
    thresholds = np.percentile(preds, 95, axis=0)  # Seuil au 95e percentile
    thresholds = np.clip(thresholds, 0.3, 0.7)  # Bornage des seuils
    
    preds = (preds > thresholds).astype(int)
    
    # Filtrage des classes sans positifs
    valid_classes = np.where(truths.sum(0) > 0)[0]
    
    return f1_score(
        truths[:, valid_classes], 
        preds[:, valid_classes], 
        average='macro', 
        zero_division=0
    )

# 7. Boucle d'entraînement avec early stopping
best_f1 = 0
patience_counter = 0

for epoch in range(100):
    model.train()
    total_loss = 0
    
    for batch in tqdm(train_loader, desc=f"Epoch {epoch}"):
        optimizer.zero_grad()
        out = model(batch)
        loss = loss_fn(out, batch.y)
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
        
        optimizer.step()
        total_loss += loss.item()
    
    # Validation
    val_f1 = evaluate(test_loader)
    scheduler.step(val_f1)
    
    # Early stopping
    if val_f1 > best_f1:
        best_f1 = val_f1
        patience_counter = 0
        torch.save(model.state_dict(), "best_model.pth")
    else:
        patience_counter += 1
    
    print(f"Epoch {epoch} | Loss: {total_loss/len(train_loader):.4f} | Val F1: {val_f1:.4f}")
    
    if patience_counter >= 10:
        print("Early stopping!")
        break

NameError: name 'df' is not defined

In [1]:
import torch
from tqdm import tqdm
from torch_geometric.data import Data, DataLoader
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch_geometric.nn import GCNConv, GATConv, GlobalAttention
from sklearn.metrics import f1_score
import numpy as np
import pandas as pd

In [3]:
df = pd.read_csv("1000_assembly_feature.csv")
labels = torch.tensor(df.drop(columns=['name']).values, dtype=torch.float)  # Shape [num_graphs, 453]

# 2. Initialisation des données
dataset = torch.load("graphs_pyg.pkl", weights_only=False)
dataset = dataset[:100]  # Votre dataset
train_data = dataset[:int(0.8 * len(dataset))]
test_data = dataset[int(0.8 * len(dataset)):]

train_loader = DataLoader(train_data, batch_size=8, shuffle=True)
test_loader = DataLoader(test_data, batch_size=8, shuffle=True)

# 3. Initialisation du modèle et de la loss
label_counts = df.drop(columns='name').sum(axis=0)
pos_weights = (len(df) - label_counts) / (label_counts + 1e-6)
pos_weights = np.clip(pos_weights, a_min=0, a_max=50)
pos_weights = torch.tensor(pos_weights.values, dtype=torch.float)
device = torch.device('mps')

/opt/anaconda3/lib/python3.12/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


In [4]:
import torch
from tqdm import tqdm
from torch_geometric.data import Data, DataLoader
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch_geometric.nn import GCNConv, GATConv, GlobalAttention
from sklearn.metrics import f1_score
import numpy as np

# Version ultra-optimisée du modèle
class LightweightGraphClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        
        # Pooling simplifié
        self.pool = GlobalAttention(nn.Linear(hidden_dim, 1))
        
        # Classificateur minimaliste
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        x = F.relu(self.bn1(self.conv1(x, edge_index)))
        x = F.relu(self.conv2(x, edge_index))
        x = self.pool(x, batch)
        return self.classifier(x)

# Loss function simplifiée
class FastFocalLoss(nn.Module):
    def __init__(self, pos_weights=None):
        super().__init__()
        self.pos_weights = pos_weights

    def forward(self, inputs, targets):
        return F.binary_cross_entropy_with_logits(
            inputs, targets, 
            pos_weight=self.pos_weights
        )

# Configuration des DataLoaders pour MPS
train_loader = DataLoader(
    train_data,
    batch_size=16,  # Batch size réduit
    shuffle=True,
    num_workers=0,  # Désactivé pour MPS
    pin_memory=False
)

test_loader = DataLoader(
    test_data,
    batch_size=16,
    num_workers=0,
    pin_memory=False
)

# Initialisation modèle léger
model = LightweightGraphClassifier(
    input_dim=10, 
    hidden_dim=64,  # Taille réduite
    output_dim=453
).to(device)

# Configuration optimiseur
optimizer = optim.AdamW(model.parameters(), lr=0.002, weight_decay=1e-4)
loss_fn = FastFocalLoss(pos_weights=pos_weights.to(device))

# Fonction d'évaluation accélérée
def fast_evaluate(loader):
    model.eval()
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            out = model(batch)
            all_preds.append(torch.sigmoid(out).cpu())
            all_targets.append(batch.y.cpu())
    
    preds = torch.cat(all_preds).numpy()
    truths = torch.cat(all_targets).numpy()
    return f1_score((preds > 0.5).astype(int), truths, average='macro')

# Boucle d'entraînement optimisée
best_f1 = 0
for epoch in range(20):  # Nombre d'epochs réduit
    model.train()
    total_loss = 0
    
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch)
        loss = loss_fn(out, batch.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    val_f1 = fast_evaluate(test_loader)
    print(f"Epoch {epoch} | Loss: {total_loss/len(train_loader):.4f} | Val F1: {val_f1:.4f}")
    
    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), "best_model.pth")
    else:
        if epoch >= 5:  # Early stopping rapide
            print("Early stopping!")
            break

/opt/anaconda3/lib/python3.12/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)
/opt/anaconda3/lib/python3.12/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'nn.glob.GlobalAttention' is deprecated, use 'nn.aggr.AttentionalAggregation' instead
  warnings.warn(out)


: 